# <a href="https://girafe.ai/" target="_blank" rel="noopener noreferrer"><img src="https://raw.githubusercontent.com/girafe-ai/ml-course/7096a5df4cada5ee651be1e3215c2f7fb8a7e0bf/logo_margin.svg" alt="girafe-ai logo" width="150px" align="left"></a> [MLOps course](https://github.com/girafe-ai/mlops) <a class="tocSkip">

# Seminar: CPU vs GPU benchmark notebook (NumPy / PyTorch / CUDA)

This file is written as a plain Python script with notebook-style cell markers.
It is meant to be pasted into a Jupyter notebook cell-by-cell or run in editors
that understand `# %%` cells.

Goals:
- compare NumPy CPU, PyTorch CPU, and PyTorch CUDA on matrix multiplication
- show why naive GPU timing is wrong
- show warmup effects
- show transfer overhead
- keep the timing code sane and readable

Notes:
- GPU timings use CUDA events + synchronization
- CPU timings use `time.perf_counter()`
- results depend heavily on hardware, BLAS library, tensor shapes, dtype, and load
- for tiny workloads the GPU may be slower


In [ ]:
import gc
import os
import platform
import statistics
import sys
import time
from dataclasses import dataclass
from typing import Callable, Iterable

import numpy as np
import torch

In [ ]:
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version (PyTorch build): {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / 1024**3:.2f} GiB")

# Optional: keep CPU thread count explicit for more stable CPU-only comparisons.
# Comment this out if you want your BLAS / PyTorch CPU backends to use all threads.
torch.set_num_threads(max(1, os.cpu_count() // 2))
print("PyTorch CPU threads:", torch.get_num_threads())

## Timing helpers

The two main rules:

1. **Warm up first**
   - first runs may include lazy initialization, kernel selection, allocator setup, caches

2. **Time GPU work correctly**
   - GPU operations are asynchronous
   - `perf_counter()` around a CUDA op usually measures dispatch time, not execution time
   - use `torch.cuda.Event` or explicit synchronization


In [ ]:
@dataclass
class BenchResult:
    label: str
    median_ms: float
    mean_ms: float
    min_ms: float
    max_ms: float
    repeats: int

    def pretty(self) -> str:
        return (
            f"{self.label:<28} "
            f"median={self.median_ms:>9.3f} ms | "
            f"mean={self.mean_ms:>9.3f} ms | "
            f"min={self.min_ms:>9.3f} ms | "
            f"max={self.max_ms:>9.3f} ms | "
            f"n={self.repeats}"
        )


def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def bench_cpu(
    fn: Callable[[], object], *, warmup: int = 3, repeats: int = 10, label: str = "cpu"
) -> BenchResult:
    for _ in range(warmup):
        fn()

    times_ms = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1_000)

    return BenchResult(
        label=label,
        median_ms=statistics.median(times_ms),
        mean_ms=statistics.mean(times_ms),
        min_ms=min(times_ms),
        max_ms=max(times_ms),
        repeats=repeats,
    )


def bench_cuda_event(
    fn: Callable[[], object], *, warmup: int = 5, repeats: int = 20, label: str = "cuda"
) -> BenchResult:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")

    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    times_ms = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        times_ms.append(start.elapsed_time(end))

    return BenchResult(
        label=label,
        median_ms=statistics.median(times_ms),
        mean_ms=statistics.mean(times_ms),
        min_ms=min(times_ms),
        max_ms=max(times_ms),
        repeats=repeats,
    )


def bench_cuda_sync_perf_counter(
    fn: Callable[[], object],
    *,
    warmup: int = 5,
    repeats: int = 20,
    label: str = "cuda_sync_perf_counter",
) -> BenchResult:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")

    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    times_ms = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1_000)

    return BenchResult(
        label=label,
        median_ms=statistics.median(times_ms),
        mean_ms=statistics.mean(times_ms),
        min_ms=min(times_ms),
        max_ms=max(times_ms),
        repeats=repeats,
    )


def print_results(results: Iterable[BenchResult]) -> None:
    for r in results:
        print(r.pretty())

## Helper data builders

We will mostly use `float32`, since it is common for deep learning and supported well on GPUs.
For strict CPU-vs-GPU fairness, keep dtype and shapes identical.


In [ ]:
def make_numpy_mats(n: int, dtype=np.float32, seed: int = 0):
    rng = np.random.default_rng(seed)
    a = rng.standard_normal((n, n), dtype=dtype)
    b = rng.standard_normal((n, n), dtype=dtype)
    return a, b


def make_torch_mats(
    n: int, *, device: str, dtype: torch.dtype = torch.float32, seed: int = 0
):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    a = torch.randn((n, n), device=device, dtype=dtype, generator=g)
    b = torch.randn((n, n), device=device, dtype=dtype, generator=g)
    return a, b

## Example 1 — naive GPU timing is wrong

This cell demonstrates the classic trap:

- `perf_counter()` around a CUDA op measures Python dispatch time unless you synchronize
- the correctly timed version is often **much** larger

In [ ]:
if torch.cuda.is_available():
    cleanup_cuda()
    n = 4096
    a_gpu, b_gpu = make_torch_mats(n, device="cuda", dtype=torch.float32)

    def matmul_gpu():
        return a_gpu @ b_gpu

    # Wrong: measures mostly launch/dispatch overhead.
    t0 = time.perf_counter()
    _ = matmul_gpu()
    t1 = time.perf_counter()
    print(f"Naive perf_counter around CUDA op: {(t1 - t0) * 1000:.3f} ms")

    # Right: synchronize before reading the wall clock.
    synced = bench_cuda_sync_perf_counter(
        matmul_gpu, warmup=3, repeats=10, label="CUDA matmul (synced)"
    )

    # Also right: CUDA events.
    events = bench_cuda_event(
        matmul_gpu, warmup=3, repeats=10, label="CUDA matmul (events)"
    )

    print_results([synced, events])
else:
    print("CUDA not available; skip this cell.")

## Example 2 — warmup matters

First iterations may be weirdly slow because libraries initialize lazily.
This is especially visible on GPU, but CPU code can also have warmup effects.

In [ ]:
n = 2048
x_cpu, y_cpu = make_torch_mats(n, device="cpu")

print("PyTorch CPU matmul warmup trace:")
for i in range(6):
    t0 = time.perf_counter()
    _ = x_cpu @ y_cpu
    t1 = time.perf_counter()
    print(f"run {i}: {(t1 - t0) * 1000:.3f} ms")

if torch.cuda.is_available():
    cleanup_cuda()
    x_gpu, y_gpu = make_torch_mats(n, device="cuda")
    print("\nPyTorch CUDA matmul warmup trace:")
    for i in range(6):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        _ = x_gpu @ y_gpu
        end.record()
        torch.cuda.synchronize()
        print(f"run {i}: {start.elapsed_time(end):.3f} ms")

## Example 3 — NumPy CPU vs PyTorch CPU vs PyTorch CUDA matmul

This is the main comparison. We benchmark square matrix multiplication at multiple sizes.
The typical pattern:

- small sizes: CPU can be competitive or faster
- medium/large sizes: GPU often wins decisively
- exact crossover depends on hardware and BLAS backend

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]
all_rows = []

for n in sizes:
    print(f"\n=== n = {n} ===")

    # NumPy CPU
    a_np, b_np = make_numpy_mats(n)
    res_np = bench_cpu(lambda: a_np @ b_np, warmup=2, repeats=7, label="NumPy CPU")
    print(res_np.pretty())
    all_rows.append((n, "numpy_cpu", res_np.median_ms))

    # PyTorch CPU
    a_t_cpu, b_t_cpu = make_torch_mats(n, device="cpu")
    res_t_cpu = bench_cpu(
        lambda: a_t_cpu @ b_t_cpu, warmup=2, repeats=7, label="PyTorch CPU"
    )
    print(res_t_cpu.pretty())
    all_rows.append((n, "torch_cpu", res_t_cpu.median_ms))

    # PyTorch CUDA
    if torch.cuda.is_available():
        cleanup_cuda()
        a_t_gpu, b_t_gpu = make_torch_mats(n, device="cuda")
        res_t_gpu = bench_cuda_event(
            lambda: a_t_gpu @ b_t_gpu, warmup=3, repeats=10, label="PyTorch CUDA"
        )
        print(res_t_gpu.pretty())
        all_rows.append((n, "torch_cuda", res_t_gpu.median_ms))


# %%
# Optional: present the summary as a compact table.
summary = {}
for n, backend, median_ms in all_rows:
    summary.setdefault(n, {})[backend] = median_ms

print("\nSummary table (median ms):")
header = ["n", "numpy_cpu", "torch_cpu", "torch_cuda"]
print(f"{header[0]:>8} {header[1]:>14} {header[2]:>14} {header[3]:>14}")
for n in sizes:
    row = summary.get(n, {})
    print(
        f"{n:>8} "
        f"{row.get('numpy_cpu', float('nan')):>14.3f} "
        f"{row.get('torch_cpu', float('nan')):>14.3f} "
        f"{row.get('torch_cuda', float('nan')):>14.3f}"
    )

## Example 4 — transfer overhead can dominate

Here we separate two scenarios:

1. **compute-only on GPU**: tensors already live on the GPU
2. **end-to-end GPU**: copy inputs to GPU, compute, copy result back

The second one is often what beginners accidentally benchmark.

In [ ]:
if torch.cuda.is_available():
    cleanup_cuda()
    n = 2048

    # Host tensors
    a_cpu, b_cpu = make_torch_mats(n, device="cpu")

    # Preloaded device tensors for compute-only timing
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()

    compute_only = bench_cuda_event(
        lambda: a_gpu @ b_gpu, warmup=3, repeats=10, label="GPU compute only"
    )

    def end_to_end_gpu():
        a = a_cpu.cuda(non_blocking=False)
        b = b_cpu.cuda(non_blocking=False)
        c = a @ b
        return c.cpu()

    end_to_end = bench_cuda_sync_perf_counter(
        end_to_end_gpu,
        warmup=2,
        repeats=7,
        label="GPU copy->compute->copy",
    )

    cpu_only = bench_cpu(lambda: a_cpu @ b_cpu, warmup=2, repeats=7, label="CPU only")

    print_results([cpu_only, compute_only, end_to_end])
else:
    print("CUDA not available; skip this cell.")

## Example 5 — repeated work: keep data on the GPU

This is the practical lesson. If you do many operations, moving data once and reusing it
is usually much better than bouncing tensors between host and device every iteration.

In [ ]:
if torch.cuda.is_available():
    cleanup_cuda()
    n = 1024
    steps = 50

    a_cpu, b_cpu = make_torch_mats(n, device="cpu")

    def bad_pattern():
        out = None
        for _ in range(steps):
            a = a_cpu.cuda()
            b = b_cpu.cuda()
            out = (a @ b).cpu()
        return out

    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()

    def good_pattern():
        out = None
        for _ in range(steps):
            out = a_gpu @ b_gpu
        return out

    bad = bench_cuda_sync_perf_counter(
        bad_pattern, warmup=1, repeats=5, label="Bad: move every step"
    )
    good = bench_cuda_event(
        good_pattern, warmup=2, repeats=10, label="Good: keep on GPU"
    )

    print_results([bad, good])
else:
    print("CUDA not available; skip this cell.")

## Example 6 — optional: pointwise operations and `torch.compile`

This cell is optional and mildly more advanced.
It demonstrates that many small ops can suffer from launch / Python overhead.
`torch.compile` may help by reducing overhead and fusing work.

Skip this cell if you want the simplest possible notebook.

In [ ]:
if torch.cuda.is_available() and hasattr(torch, "compile"):
    cleanup_cuda()

    x = torch.randn(8_000_000, device="cuda")

    def eager_fn(x):
        return torch.sin(x) + 0.1 * x * x + torch.relu(x)

    compiled_fn = torch.compile(eager_fn)

    eager_res = bench_cuda_event(
        lambda: eager_fn(x), warmup=5, repeats=30, label="Eager pointwise chain"
    )
    compiled_res = bench_cuda_event(
        lambda: compiled_fn(x), warmup=5, repeats=30, label="Compiled pointwise chain"
    )
    print_results([eager_res, compiled_res])
else:
    print("`torch.compile` demo skipped.")

## Tiny checklist for fair-ish benchmarks

- compare the same dtype and shape
- warm up before measuring
- use median over multiple runs
- synchronize CUDA before reading wall time, or use CUDA events
- separate compute-only timing from transfer-inclusive timing
- avoid benchmarking while the machine is under heavy load
- treat single-run numbers as gossip, not truth

## Self-check questions

- Why is naive CUDA timing fake?
- Why are first runs slower?
- At what size does the GPU start winning?
- Why can end-to-end GPU be slower than compute-only GPU?
- Why is keeping tensors on the GPU across repeated operations important?

That usually forces the correct mental model into place.